In [1]:
from copy import deepcopy

# 9x9 数独，0 表示空格
Grid = list[list[int]]

def print_grid(grid: Grid):
    for i, row in enumerate(grid):
        print(" ".join(str(x) if x != 0 else "." for x in row))
        if i % 3 == 2 and i != 8:
            print("-" * 21)


def get_block_index(row, col):
    return (row // 3) * 3 + (col // 3)


def init_candidates(grid: Grid):
    """
    为每个格子初始化候选集合：candidates[(r, c)] = {1..9} 或已缩小的集合
    """
    candidates = {}
    rows = [set() for _ in range(9)]
    cols = [set() for _ in range(9)]
    blocks = [set() for _ in range(9)]

    # 先记录已有数字
    for r in range(9):
        for c in range(9):
            val = grid[r][c]
            if val != 0:
                rows[r].add(val)
                cols[c].add(val)
                blocks[get_block_index(r, c)].add(val)

    # 初始化候选
    for r in range(9):
        for c in range(9):
            if grid[r][c] == 0:
                used = rows[r] | cols[c] | blocks[get_block_index(r, c)]
                candidates[(r, c)] = set(range(1, 10)) - used
            else:
                candidates[(r, c)] = set()

    return candidates


def find_mrv_cell(candidates):
    """
    MRV：找到候选数最少的未填格子
    """
    mrv_cell = None
    mrv_size = 10
    for (r, c), cand in candidates.items():
        if len(cand) > 0 and len(cand) < mrv_size:
            mrv_size = len(cand)
            mrv_cell = (r, c)
    return mrv_cell


def propagate(grid: Grid, candidates):
    """
    简单约束传播：
    - 若某格子只剩一个候选数，则直接填入
    - 填入后更新相关格子的候选集
    若出现某格子候选集为空且未填值 → 冲突，返回 False
    """
    changed = True
    while changed:
        changed = False
        singles = [(cell, next(iter(cand)))
                   for cell, cand in candidates.items()
                   if len(cand) == 1]

        if not singles:
            break

        for (r, c), val in singles:
            if grid[r][c] != 0:
                continue
            grid[r][c] = val
            candidates[(r, c)] = set()
            changed = True

            # 更新同行、同列、同宫候选
            for rr in range(9):
                if (rr, c) in candidates and val in candidates[(rr, c)]:
                    candidates[(rr, c)].remove(val)
                    if len(candidates[(rr, c)]) == 0 and grid[rr][c] == 0:
                        return False

            for cc in range(9):
                if (r, cc) in candidates and val in candidates[(r, cc)]:
                    candidates[(r, cc)].remove(val)
                    if len(candidates[(r, cc)]) == 0 and grid[r][cc] == 0:
                        return False

            br = (r // 3) * 3
            bc = (c // 3) * 3
            for rr in range(br, br + 3):
                for cc in range(bc, bc + 3):
                    if (rr, cc) in candidates and val in candidates[(rr, cc)]:
                        candidates[(rr, cc)].remove(val)
                        if len(candidates[(rr, cc)]) == 0 and grid[rr][cc] == 0:
                            return False

    return True


def is_complete(grid: Grid):
    return all(all(cell != 0 for cell in row) for row in grid)


def solve_sudoku(grid: Grid):
    """
    回溯 + MRV + 前向检查 + 约束传播
    """
    candidates = init_candidates(grid)
    if not propagate(grid, candidates):
        return None

    def backtrack(grid, candidates):
        if is_complete(grid):
            return grid

        cell = find_mrv_cell(candidates)
        if cell is None:
            return None

        r, c = cell
        for val in sorted(candidates[(r, c)]):
            new_grid = deepcopy(grid)
            new_cand = deepcopy(candidates)

            new_grid[r][c] = val
            new_cand[(r, c)] = set()

            # 前向检查：更新相关候选
            for rr in range(9):
                if (rr, c) in new_cand and val in new_cand[(rr, c)]:
                    new_cand[(rr, c)].remove(val)
            for cc in range(9):
                if (r, cc) in new_cand and val in new_cand[(r, cc)]:
                    new_cand[(r, cc)].remove(val)
            br = (r // 3) * 3
            bc = (c // 3) * 3
            for rr in range(br, br + 3):
                for cc in range(bc, bc + 3):
                    if (rr, cc) in new_cand and val in new_cand[(rr, cc)]:
                        new_cand[(rr, cc)].remove(val)

            # 约束传播
            if not propagate(new_grid, new_cand):
                continue

            res = backtrack(new_grid, new_cand)
            if res is not None:
                return res

        return None

    return backtrack(grid, candidates)

In [2]:
puzzle = [
    [5, 3, 0, 0, 7, 0, 0, 0, 0],
    [6, 0, 0, 1, 9, 5, 0, 0, 0],
    [0, 9, 8, 0, 0, 0, 0, 6, 0],

    [8, 0, 0, 0, 6, 0, 0, 0, 3],
    [4, 0, 0, 8, 0, 3, 0, 0, 1],
    [7, 0, 0, 0, 2, 0, 0, 0, 6],

    [0, 6, 0, 0, 0, 0, 2, 8, 0],
    [0, 0, 0, 4, 1, 9, 0, 0, 5],
    [0, 0, 0, 0, 8, 0, 0, 7, 9],
]

print("Puzzle:")
print_grid(puzzle)
solution = solve_sudoku(puzzle)
print("\nSolution:")
print_grid(solution)

Puzzle:
5 3 . . 7 . . . .
6 . . 1 9 5 . . .
. 9 8 . . . . 6 .
---------------------
8 . . . 6 . . . 3
4 . . 8 . 3 . . 1
7 . . . 2 . . . 6
---------------------
. 6 . . . . 2 8 .
. . . 4 1 9 . . 5
. . . . 8 . . 7 9

Solution:
5 3 4 6 7 8 9 1 2
6 7 2 1 9 5 3 4 8
1 9 8 3 4 2 5 6 7
---------------------
8 5 9 7 6 1 4 2 3
4 2 6 8 5 3 7 9 1
7 1 3 9 2 4 8 5 6
---------------------
9 6 1 5 3 7 2 8 4
2 8 7 4 1 9 6 3 5
3 4 5 2 8 6 1 7 9
